In [26]:
import json
import re
import os

In [27]:
FOLDER_PATH = "cleaned-data"
FILE_NAME = "cleaned-data/button.json"

In [ ]:
def read_raw_data_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

raw_data = read_raw_data_json(os.path.join(FOLDER_PATH, FILE_NAME))

print('Raw file data loaded successfully')

In [ ]:
def extract_root_document(data):
    node_id = list(data['nodes'].keys())[0]

    return data['nodes'][node_id]['document']


root_document = extract_root_document(raw_data)

component_name = root_document.get('name', 'Unknown')

print(f"Root document '{component_name}' extracted successfully")

In [30]:
def extract_component_schema(document):
    schema = document.get('componentPropertyDefinitions', {})

    clean_schema = {}
    for key, val in schema.items():
        clean_key = key.split('#')[0]
        clean_key = re.sub(r'[^a-zA-Z\s]', '', clean_key).strip()

        clean_schema[clean_key] = val
    return clean_schema


component_schema = extract_component_schema(root_document)

print(f"Component schema extracted successfully with {len(component_schema)} properties")

for prop_name, prop_details in component_schema.items():
    print(f"Property: '{prop_name}' - Type: '{prop_details.get('type', 'Unknown')}' - Default vale: {prop_details.get('defaultValue', 'Unknown')}")

    if prop_details.get('type') == 'VARIANT':
        print(f"  - Variant options: {prop_details.get('variantOptions', [])}")

Component schema extracted successfully with 14 properties
Property: 'Show Left Icon' - Type: 'BOOLEAN' - Default vale: True
Property: 'Show Right Icon' - Type: 'BOOLEAN' - Default vale: True
Property: 'Left Icon' - Type: 'INSTANCE_SWAP' - Default vale: 6115:20890
Property: 'Right Icon' - Type: 'INSTANCE_SWAP' - Default vale: 6115:20890
Property: 'Icon' - Type: 'INSTANCE_SWAP' - Default vale: 6115:20890
Property: 'Text' - Type: 'VARIANT' - Default vale: False
  - Variant options: ['False', 'True']
Property: 'Severity' - Type: 'VARIANT' - Default vale: Primary
  - Variant options: ['Primary', 'Secondary', 'Success', 'Info', 'Warn', 'Help', 'Danger', 'Contrast', 'Plain']
Property: 'State' - Type: 'VARIANT' - Default vale: Idle
  - Variant options: ['Idle', 'Hover', 'Active']
Property: 'Disabled' - Type: 'VARIANT' - Default vale: False
  - Variant options: ['False', 'True']
Property: 'Icon Only' - Type: 'VARIANT' - Default vale: False
  - Variant options: ['False', 'True']
Property: 'Rais

In [31]:
def extract_variant_content(node, pruned_content):
    node_type = node.get('type', '')
    node_name = node.get('name', '').lower()

    if node_type == 'TEXT':
        text_val = node.get('characters', '')

        if 'placeholder' in node_name:
            pruned_content["placeholder"] = text_val
        elif 'value' in node_name or 'label' in node_name:
            pruned_content["value"] = text_val
        else:
            if "texts" not in pruned_content:
                pruned_content["texts"] = []
            pruned_content["texts"].append(text_val)

    elif node_type == 'INSTANCE':
        refs = node.get('componentPropertyReferences', {})

        if 'visible' in refs:
            visible_ref = refs['visible'] # z.B. "Show Left Icon#123:4" oder "Show Check#55:1"

            # 1. IDs und Emojis entfernen -> "Show Left Icon"
            clean_ref = re.sub(r'[^a-zA-Z\s]', '', visible_ref.split('#')[0]).strip()

            # 2. Das Wort "Show" entfernen, falls PrimeVue es verwendet -> "Left Icon"
            if clean_ref.startswith("Show "):
                clean_ref = clean_ref[5:]

            # 3. Leerzeichen entfernen für einen schönen JSON-Key -> "LeftIcon"
            camel_case_ref = clean_ref.replace(" ", "")

            # 4. Dynamischen Boolean-Key setzen -> pruned["content"]["hasLeftIcon"] = True
            dynamic_key = f"has{camel_case_ref}"
            pruned_content[dynamic_key] = True

    for child in node.get('children', []):
        extract_variant_content(child, pruned_content)

def clean_up_variants_with_schema(component_name, component_schema, variants):
    cleaned_variants = []

    for variant in variants:
        if variant.get('type') != 'COMPONENT':
            continue

        pruned = {
            "id": variant.get("id"),
            "componentType": component_name,
            "properties": {},
            "content": {}
        }

        raw_name = variant.get('name', '')
        pairs = raw_name.split(', ')

        for pair in pairs:
            if '=' in pair:
                key_raw, val_raw = pair.split('=', 1)

                clean_key = key_raw.split('#')[0]
                clean_key = re.sub(r'[^a-zA-Z\s]', '', clean_key).strip()
                clean_val = val_raw.strip()

                if clean_key in component_schema:
                    if clean_val.lower() == 'false':
                        clean_val = False
                    elif clean_val.lower() == 'true':
                        clean_val = True

                    pruned["properties"][clean_key] = clean_val

        extract_variant_content(variant, pruned["content"])

        cleaned_variants.append(pruned)

    return cleaned_variants

variants = root_document.get('children', [])

cleaned_up_variants = clean_up_variants_with_schema(component_name, component_schema, variants)

print(f"Cleaned up {len(cleaned_up_variants)} variants successfully")

for variant in cleaned_up_variants:
    print(f"Variant ID: {variant['id']} - Properties: {variant['properties']} - Content: {variant['content']}")

Cleaned up 655 variants successfully
Variant ID: 10:124 - Properties: {'Severity': 'Primary', 'State': 'Idle', 'Disabled': False, 'Icon Only': False, 'Raised': False, 'Rounded': False, 'Text': False, 'Outlined': False, 'Link': False} - Content: {'hasLeftIcon': True, 'texts': ['Button'], 'hasRightIcon': True}
Variant ID: 4452:50769 - Properties: {'Severity': 'Primary', 'State': 'Idle', 'Disabled': False, 'Icon Only': False, 'Raised': False, 'Rounded': False, 'Text': False, 'Outlined': False, 'Link': True} - Content: {'hasLeftIcon': True, 'texts': ['Button'], 'hasRightIcon': True}
Variant ID: 4452:51350 - Properties: {'Severity': 'Primary', 'State': 'Active', 'Disabled': False, 'Icon Only': False, 'Raised': False, 'Rounded': False, 'Text': False, 'Outlined': False, 'Link': True} - Content: {'hasLeftIcon': True, 'texts': ['Button'], 'hasRightIcon': True}
Variant ID: 4452:51054 - Properties: {'Severity': 'Primary', 'State': 'Hover', 'Disabled': False, 'Icon Only': False, 'Raised': False, '

In [ ]:
if cleaned_up_variants:
    if not os.path.exists(FOLDER_PATH):
        os.makedirs(FOLDER_PATH)

    file_path = os.path.join(FOLDER_PATH, f"{component_name}-component-variants.json")

    with open(file_path, "w", encoding="utf-8") as json_file:
        json.dump(cleaned_up_variants, json_file, indent=4, ensure_ascii=False)

    print(f"Data successfully saved to {file_path}")
else:
    print("No data to save")